# Activity 6: Regularization with Ridge, LASSO and Elastic Net

**Tuesday · Block 1** · Kernel: `nwcst` · About 20–30 minutes

This is the hands-on part of the regularization lecture. You will estimate the three
penalized regressions from the slides (Ridge, LASSO and Elastic Net) and compare them with
plain OLS and with two naive benchmarks. The same steps repeat in Blocks 2 and 3:

1. Split the sample into **train** and **test** (already done in the data file)
2. **Fit** the model on the train sample
3. **Predict** the test sample
4. **Score** it with the RMSE and save the result in the workshop table

Almost every cell is already written; you only run it. Cells marked **YOUR TURN** hold one
parameter that you change to see what moves. That is the point of the block: not to write the
model from scratch, but to understand what each knob does.

## Preparation

The data are the shared files for the three Tuesday blocks, already in `data/`. They were
prepared exactly as in Activity 5 (build) and in the nowcast tool: each series is seasonally
adjusted at its own frequency and turned into a growth rate (quarter on quarter for GDP,
month on month for the predictors, then averaged within each quarter). The money and credit
series are left in nominal terms, because T&T's CPI only starts in 2015 and deflating would
cut ten quarters from an already short sample.

| File | What it holds |
|---|---|
| `blocks_quarterly.csv` | Real GDP growth (`RGDP0000`), a `sample` column (`train` / `test`) and 15 predictors |
| `blocks_meta.csv` | Ticker, short label, group and description of each predictor |

The split is fixed so that every block, in Python and in R, is scored on the same quarters:
**train** = 2013Q1 to 2023Q1 (41 quarters), **test** = 2023Q2 to 2026Q1 (12 quarters).

Run as is; there is nothing to change here.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Relative folders: run this notebook from activity/ (or sols/)
PATH_DATA = os.path.join("..", "data")
PATH_OUTPUT = os.path.join("..", "output")
os.makedirs(PATH_OUTPUT, exist_ok=True)

# Cumulative results table, shared by every block and by the R activities
RESULTS_FILE = os.path.join(PATH_OUTPUT, "workshop_results.csv")
LANGUAGE = "Python"

target_variable = "RGDP0000"

data = pd.read_csv(os.path.join(PATH_DATA, "blocks_quarterly.csv"),
                   parse_dates=["date"], index_col="date")
meta = pd.read_csv(os.path.join(PATH_DATA, "blocks_meta.csv"))

predictors = [c for c in data.columns if c not in (target_variable, "sample")]
LABELS = meta.set_index("ticker")["label"].to_dict()
labels = [LABELS.get(c, c) for c in predictors]

train = data[data["sample"] == "train"]
test = data[data["sample"] == "test"]

# Raw (unscaled) predictors
X_train_raw = train[predictors]
X_test_raw = test[predictors]
y_train = train[target_variable]
y_obs = test[target_variable]

# Standardize with the TRAIN mean and standard deviation only (the test quarters stay unseen)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train_raw)
X_train = pd.DataFrame(scaler.transform(X_train_raw), index=train.index, columns=predictors)
X_test = pd.DataFrame(scaler.transform(X_test_raw), index=test.index, columns=predictors)

The helper functions below (`rmse`, `comparison_table`, `save_result`, `results_table`,
`plot_observed_predicted`, `data_summary`) are the same in all three blocks. Run and move on.

In [ ]:
from datetime import datetime

RESULT_COLS = ["block", "activity", "language", "model", "rmse_test", "notes"]


def rmse(observed, predicted):
    """Root mean squared error."""
    return float(np.sqrt(np.mean((np.asarray(observed) - np.asarray(predicted)) ** 2)))


def comparison_table(predicted, name="Predicted"):
    """Observed vs predicted on the test sample, with the error in percentage points."""
    t = pd.DataFrame({"Observed": y_obs, name: np.asarray(predicted)}, index=y_obs.index)
    t["Error (pp)"] = t[name] - t["Observed"]
    return t.round(3)


def save_result(block, activity, model, rmse_value, notes=""):
    """Add or replace one row of the workshop results table.

    A row is identified by (activity, language, model): re-running a cell replaces its
    row instead of duplicating it. Rows written by the R activities are kept untouched.
    """
    row = pd.DataFrame([{
        "block": int(block), "activity": activity, "language": LANGUAGE,
        "model": model, "rmse_test": round(float(rmse_value), 4), "notes": notes,
    }])
    if os.path.exists(RESULTS_FILE) and os.path.getsize(RESULTS_FILE) > 0:
        prev = pd.read_csv(RESULTS_FILE)
        same = ((prev["activity"] == activity) & (prev["language"] == LANGUAGE)
                & (prev["model"] == model))
        row = pd.concat([prev[~same], row], ignore_index=True)
    os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
    row[RESULT_COLS].to_csv(RESULTS_FILE, index=False)
    print(f"Saved: {model} ({activity}, {LANGUAGE})  RMSE = {float(rmse_value):.4f}")


def results_table():
    """Everything saved so far, by every block and language, best to worst."""
    if not os.path.exists(RESULTS_FILE) or os.path.getsize(RESULTS_FILE) == 0:
        print("No results saved yet.")
        return pd.DataFrame(columns=RESULT_COLS)
    t = pd.read_csv(RESULTS_FILE)
    t = t.sort_values("rmse_test").reset_index(drop=True)
    print(t.to_string())
    return t


def plot_observed_predicted(predicted, name, color="#C0392B", ax=None):
    """Observed vs predicted GDP growth on the test sample."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))
    ax.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2, label="Observed")
    ax.plot(y_obs.index, np.asarray(predicted), "s--", color=color, lw=2, label=name)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylabel("Real GDP growth, q/q (%)")
    ax.set_title(f"Observed vs predicted: {name}")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    return ax


def data_summary():
    """Print the state of the data, to check that everything loaded."""
    print("=" * 66)
    print("DATA READY FOR THE ACTIVITY")
    print("=" * 66)
    print(f"Target     : {target_variable} (real GDP growth, q/q %)")
    print(f"Predictors : {len(predictors)}")
    print(f"Train      : {train.index.min():%Y-%m} to {train.index.max():%Y-%m} "
          f"({len(train)} quarters)")
    print(f"Test       : {test.index.min():%Y-%m} to {test.index.max():%Y-%m} "
          f"({len(test)} quarters)")
    print("=" * 66)
    return pd.DataFrame({
        "label": labels,
        "group": meta.set_index("ticker").loc[predictors, "group"].values,
        "train mean": X_train_raw.mean().values,
        "train std": X_train_raw.std().values,
    }, index=predictors).round(2)

In [ ]:
data_summary()

The predictors move on very different scales: Brent and cement sales swing by tens of percent
a quarter, credit and deposits by a fraction of a percent. A penalty on the size of a
coefficient is only fair if all coefficients are measured in the same units, so **X is
standardized** (mean 0, standard deviation 1) using the train sample only. From here on every
coefficient reads as "percentage points of GDP growth per one standard deviation of the
predictor", and coefficients can be compared across variables.

### A word on 2020

One quarter dominates the train sample. In 2020Q2 GDP fell by 12.3% in a single quarter; the
next-largest move is about 5%. With squared errors, that one quarter pulls hard on every
model below. We keep it in (it really happened, and the test sample does not contain anything
like it), but keep it in mind when you read the results.

In [ ]:
dev2 = (y_train - y_train.mean()) ** 2
share = dev2.sort_values(ascending=False) / dev2.sum()
print(f"Train-sample variance of GDP growth: {y_train.var():.2f}")
print(f"  without the four 2020 quarters  : {y_train[y_train.index.year != 2020].var():.2f}")
print("\nShare of the total sum of squares contributed by each quarter (top 5):")
print((share.head(5) * 100).round(1).astype(str) + " %")

## Two naive benchmarks

Before fitting anything clever we need numbers to beat. Two are standard in nowcasting:

- **Train mean**: predict every test quarter with the average growth of the train sample.
- **AR(1)**: regress GDP growth on its own value one quarter earlier, on the train sample,
  then predict each test quarter from the previous quarter's actual growth. This is a
  generous benchmark, since it assumes last quarter's GDP is already published.

If a model with 15 indicators cannot beat these, the indicators are not adding information.

In [ ]:
import statsmodels.api as sm

# 1) Train mean
pred_mean = np.repeat(y_train.mean(), len(y_obs))
rmse_mean = rmse(y_obs, pred_mean)

# 2) AR(1): y_t = c + phi * y_{t-1}, estimated on the train sample
gdp = data[target_variable]
gdp_lag = gdp.shift(1)
ar_train = pd.concat([y_train, gdp_lag.loc[train.index]], axis=1, keys=["y", "lag"]).dropna()
ar1 = sm.OLS(ar_train["y"], sm.add_constant(ar_train["lag"])).fit()
pred_ar1 = ar1.params["const"] + ar1.params["lag"] * gdp_lag.loc[test.index]
rmse_ar1 = rmse(y_obs, pred_ar1)

print(f"AR(1) coefficients: c = {ar1.params['const']:.3f}, phi = {ar1.params['lag']:.3f}")
print(f"RMSE, train mean : {rmse_mean:.4f}")
print(f"RMSE, AR(1)      : {rmse_ar1:.4f}")

save_result(0, "act6", "Mean (train)", rmse_mean, "naive benchmark")
save_result(0, "act6", "AR(1)", rmse_ar1, f"phi={ar1.params['lag']:.3f}, uses last quarter's actual GDP")

## Starting point: OLS

Plain OLS on all 15 predictors is the reference that every penalized model shrinks away from.

In [ ]:
from sklearn.linear_model import LinearRegression

ols = LinearRegression().fit(X_train, y_train)
pred_ols = ols.predict(X_test)

rmse_ols = rmse(y_obs, pred_ols)
print(f"RMSE, OLS: {rmse_ols:.4f}   (train mean: {rmse_mean:.4f}, AR(1): {rmse_ar1:.4f})")

comparison_table(pred_ols, "OLS")

### The full OLS diagnostics

`LinearRegression` from sklearn only returns the fitted model. To see standard errors,
t statistics, p-values and R² we use `statsmodels`. It is the same model with the full
diagnostics, which we will not have for Ridge, LASSO or the trees, because those models do not
report significance in this way.

Compare the in-sample R² here with the test RMSE above: 15 predictors and 41 quarters leave
plenty of room to fit noise.

In [ ]:
X_train_const = sm.add_constant(X_train.rename(columns=LABELS))
ols_sm = sm.OLS(y_train, X_train_const).fit()
print(ols_sm.summary())

In [ ]:
plot_observed_predicted(pred_ols, "OLS")
save_result(1, "act6", "OLS", rmse_ols, "no penalty, standardized X")

## Ridge

Ridge adds the L2 penalty, which shrinks all coefficients toward zero in proportion but never
sets any of them exactly to zero. `RidgeCV` tries a grid of penalty values and keeps the one
with the lowest cross-validation error.

| Parameter | Effect |
|---|---|
| `alphas` | Grid of penalties to try. Large values shrink more; values near zero give back OLS. |
| `cv` | How the train sample is split for cross-validation. |

**Why `TimeSeriesSplit` and not shuffled K-fold:** with time series, the model should always
be validated on quarters that come *after* the ones it was trained on, as in a real nowcast,
so `TimeSeriesSplit` grows the training window forward and never lets the future leak into
the past.

The fitted model's `.alpha_` attribute tells you which penalty won.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit

cv = TimeSeriesSplit(n_splits=5)
grid = np.logspace(-2, 3, 30)

ridge = RidgeCV(alphas=grid, cv=cv, scoring="neg_root_mean_squared_error").fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)

rmse_ridge = rmse(y_obs, pred_ridge)
print(f"Alpha chosen by cross-validation: {ridge.alpha_:.2f}")
print(f"RMSE, Ridge: {rmse_ridge:.4f}   (OLS: {rmse_ols:.4f})")

save_result(1, "act6", "Ridge", rmse_ridge, f"alpha={ridge.alpha_:.2f}, TimeSeriesSplit(5)")

### Chart 1: The cross-validation curve

Where did that alpha come from? The horizontal axis is the penalty (log scale) and the
vertical axis is the cross-validation error. The dashed line marks what `RidgeCV` picked.

The textbook picture is a U: on the left there is little penalty and the model overfits; on
the right the penalty is so strong that the model stops following the data, and the minimum in
between is the bias-variance trade-off. On T&T data the U is lopsided. The error falls steeply
as the penalty grows and then flattens instead of rising, because a heavily shrunk Ridge
predicts almost the train mean for every quarter, and with these predictors the mean is hard
to beat. The chosen alpha sits at the start of that flat floor.

The thin grey lines are the five folds. One of them validates on 2020, and its error is far
above the others: the average is pulled up by that single fold.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

alphas = np.logspace(-2, 3, 60)
cv_by_fold = np.array([
    -cross_val_score(Ridge(alpha=a), X_train, y_train,
                     scoring="neg_root_mean_squared_error", cv=cv)
    for a in alphas
])
cv_errors = cv_by_fold.mean(axis=1)

fig, ax = plt.subplots(figsize=(9, 4.5))
for k in range(cv_by_fold.shape[1]):
    ax.plot(alphas, cv_by_fold[:, k], color="grey", lw=0.8, alpha=0.5,
            label="Single fold" if k == 0 else None)
ax.plot(alphas, cv_errors, color="#2C3E50", lw=2.5, label="Average over folds")
ax.axvline(ridge.alpha_, color="#C0392B", ls="--", lw=2,
           label=f"Chosen alpha = {ridge.alpha_:.1f}")
ax.set_xscale("log")
ax.set_xlabel("Penalty (alpha, log scale)")
ax.set_ylabel("Cross-validation RMSE")
ax.set_title("Cross-validation curve: Ridge")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()

# Which quarters does each fold validate on?
for k, (_, val) in enumerate(cv.split(X_train)):
    print(f"Fold {k + 1}: validates on {X_train.index[val[0]]:%Y-%m} to {X_train.index[val[-1]]:%Y-%m}")

## LASSO

LASSO uses the L1 penalty, which, unlike Ridge, sets coefficients exactly to zero. That is
why it is said to select variables: the final model drops whole predictors instead of keeping
all of them shrunk.

| Parameter | Effect |
|---|---|
| `alpha` | Strength of the L1 penalty. Higher means more zeros and a simpler model; lower gets closer to OLS. |

### Exploring the parameter

Instead of editing `alpha` by hand and re-running the cell again and again, the cell below
tries seven typical values at once and tabulates the test RMSE and how many variables survive
each one. You see the whole pattern without losing track of what you tried.

In [ ]:
from sklearn.linear_model import Lasso

alphas_explore = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]

exploration = []
for a in alphas_explore:
    m = Lasso(alpha=a, max_iter=50000).fit(X_train, y_train)
    exploration.append({
        "alpha": a,
        "RMSE (test)": round(rmse(y_obs, m.predict(X_test)), 4),
        "Surviving variables": int((np.abs(m.coef_) > 1e-10).sum()),
    })

exploration = pd.DataFrame(exploration)
exploration

### YOUR TURN: fix the final model

From the table above, pick the `alpha` with the best RMSE (or the one you think best balances
error and number of variables) and set it here. This is the LASSO that is saved as the
block's official result.

A caution: choosing alpha by looking at the test RMSE uses the test sample to tune the model,
which in a strict evaluation is cheating. It is fine for learning what the knob does. In
production, the penalty is chosen by cross-validation on the train sample only; the cell also
prints what `LassoCV` with the same `TimeSeriesSplit` would pick.

In [ ]:
# ============ YOUR TURN ============
# Copy here the alpha you chose from the exploration table above.
alpha_lasso = 0.1
# ===================================

lasso = Lasso(alpha=alpha_lasso, max_iter=50000).fit(X_train, y_train)
pred_lasso = lasso.predict(X_test)
rmse_lasso = rmse(y_obs, pred_lasso)

from sklearn.linear_model import LassoCV
lasso_cv = LassoCV(cv=cv, max_iter=50000).fit(X_train, y_train)

print(f"RMSE, LASSO: {rmse_lasso:.4f}   (OLS: {rmse_ols:.4f})")
print(f"Surviving variables: {(np.abs(lasso.coef_) > 1e-10).sum()} of {len(predictors)}")
print(f"For reference, LassoCV with TimeSeriesSplit picks alpha = {lasso_cv.alpha_:.3f}")

coefs = pd.DataFrame({
    "Variable": labels,
    "Coefficient": lasso.coef_.round(4),
    "Survives": np.where(np.abs(lasso.coef_) > 1e-10, "Yes", "No"),
})
coefs.sort_values("Coefficient", key=np.abs, ascending=False)

In [ ]:
save_result(1, "act6", "LASSO", rmse_lasso, f"alpha={alpha_lasso}")

### Chart 2: The regularization path

This is the central chart of the block. Each line is one coefficient, and the horizontal axis
is the penalty growing from left to right. At the far left there is no penalty and the
coefficients are the OLS values; moving right, the penalty tightens.

The difference between the two panels sums up the lecture. In LASSO the lines hit the zero
line and stay there: the variable has left the model. In Ridge they approach zero without
ever touching it, because the L2 penalty shrinks but does not remove. (The two penalties work
on different scales, so each panel has its own horizontal range.)

Question for the group: which T&T series is the last one to die in the LASSO panel? Because X
is standardized, the order is not driven by how volatile each series is; the last survivor is
the one the model finds hardest to replace.

In [ ]:
path_lasso = np.logspace(-3, 0.5, 80)
path_ridge = np.logspace(-2, 4, 80)

coef_lasso = np.array([Lasso(alpha=a, max_iter=50000).fit(X_train, y_train).coef_ for a in path_lasso])
coef_ridge = np.array([Ridge(alpha=a).fit(X_train, y_train).coef_ for a in path_ridge])

colors = plt.cm.tab20(np.linspace(0, 1, len(labels)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for j, name in enumerate(labels):
    axes[0].plot(path_lasso, coef_lasso[:, j], lw=2, color=colors[j], label=name)
    axes[1].plot(path_ridge, coef_ridge[:, j], lw=2, color=colors[j], label=name)

for ax, title in zip(axes, ["LASSO (L1 penalty)", "Ridge (L2 penalty)"]):
    ax.set_xscale("log")
    ax.axhline(0, color="black", lw=1.2)
    ax.set_xlabel("Penalty (alpha, log scale)")
    ax.set_title(title)
    ax.grid(alpha=0.25)
axes[0].axvline(alpha_lasso, color="#C0392B", ls="--", lw=1.5)

axes[0].set_ylabel("Coefficient (pp of GDP growth per 1 s.d.)")
axes[1].legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.suptitle("Regularization path: who hits zero and who only gets close", y=1.02)
plt.tight_layout()

# Order in which variables leave the LASSO model
alive = np.abs(coef_lasso) > 1e-10
last_alive = [path_lasso[alive[:, j]].max() if alive[:, j].any() else 0 for j in range(len(labels))]
pd.Series(last_alive, index=labels, name="largest alpha at which it survives").sort_values(ascending=False).round(3)

## Elastic Net

Elastic Net mixes the two penalties. The L1 part selects variables and the L2 part stabilizes
the estimate when predictors are correlated with each other; in that case LASSO tends to keep
one of the correlated series and drop the rest fairly arbitrarily (think of the two credit
series, or deposits and M1).

There is a second reason, which does not bite with 15 variables and 41 quarters but matters a
lot in the full pipeline: when there are more predictors than observations (p > n), LASSO
cannot select more than n variables, however informative the rest are. The L2 term removes
that limit, which is why the comparison slide calls Elastic Net the sensible default for
nowcasting.

| Parameter | Effect |
|---|---|
| `alpha` | Overall strength of the penalty. |
| `l1_ratio` | The mix between the two. 1 is pure LASSO, 0 is pure Ridge. |

### Exploring the parameter

As with LASSO, the cell runs through a grid of `l1_ratio` values at a fixed `alpha` and counts
how many variables survive each mix.

In [ ]:
from sklearn.linear_model import ElasticNet

alpha_enet = 1.0
l1_ratios_explore = [0.1, 0.25, 0.5, 0.75, 0.9]

exploration_enet = []
for l1r in l1_ratios_explore:
    m = ElasticNet(alpha=alpha_enet, l1_ratio=l1r, max_iter=50000).fit(X_train, y_train)
    exploration_enet.append({
        "l1_ratio": l1r,
        "RMSE (test)": round(rmse(y_obs, m.predict(X_test)), 4),
        "Surviving variables": int((np.abs(m.coef_) > 1e-10).sum()),
    })

exploration_enet = pd.DataFrame(exploration_enet)
exploration_enet

### YOUR TURN: fix the final model

Pick the `l1_ratio` from the table above and set it here as the official version.

In [ ]:
# ============ YOUR TURN ============
# Copy here the l1_ratio you chose from the exploration table above.
l1_ratio = 0.5
# ===================================

enet = ElasticNet(alpha=alpha_enet, l1_ratio=l1_ratio, max_iter=50000).fit(X_train, y_train)
pred_enet = enet.predict(X_test)
rmse_enet = rmse(y_obs, pred_enet)

print(f"RMSE, Elastic Net: {rmse_enet:.4f}   (OLS: {rmse_ols:.4f})")
print(f"Surviving variables: {(np.abs(enet.coef_) > 1e-10).sum()} of {len(predictors)}")

save_result(1, "act6", "Elastic Net", rmse_enet, f"alpha={alpha_enet}, l1_ratio={l1_ratio}")

### Chart 3: Four models, the same variables

The coefficients of the four models side by side for each of the 15 series. Missing LASSO and
Elastic Net bars are the variables the penalty removed; the change in height from OLS is how
much each model shrank the ones it kept.

Read the chart vertically first (the same variable across models): that is the effect of the
penalty. Because X is standardized, the horizontal comparison (across variables) is also
meaningful here: a taller bar means a larger effect per standard deviation of the predictor.

In [ ]:
comparison = pd.DataFrame({
    "OLS": ols.coef_,
    "Ridge": ridge.coef_,
    "LASSO": lasso.coef_,
    "Elastic Net": enet.coef_,
}, index=labels)

fig, ax = plt.subplots(figsize=(12, 5))
width = 0.2
x = np.arange(len(labels))
palette = ["#95A5A6", "#2980B9", "#C0392B", "#27AE60"]

for i, (model, color) in enumerate(zip(comparison.columns, palette)):
    ax.bar(x + i * width - 1.5 * width, comparison[model], width, label=model, color=color)

ax.axhline(0, color="black", lw=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.set_ylabel("Coefficient (pp per 1 s.d.)")
ax.set_title("Coefficients estimated by the four models")
ax.legend(frameon=False, ncol=4)
ax.grid(alpha=0.25, axis="y")
plt.tight_layout()

comparison.round(4)

### Chart 4: The test quarters

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2.5, label="Observed")
ax.plot(y_obs.index, pred_mean, ":", color="grey", lw=1.5, label="Train mean")
ax.plot(y_obs.index, pred_ols, "s--", color="#95A5A6", lw=1.5, label="OLS")
ax.plot(y_obs.index, pred_ridge, "^--", color="#2980B9", lw=1.8, label="Ridge")
ax.plot(y_obs.index, pred_lasso, "d--", color="#C0392B", lw=1.8, label="LASSO")
ax.plot(y_obs.index, pred_enet, "v--", color="#27AE60", lw=1.8, label="Elastic Net")
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("Real GDP growth, q/q (%)")
ax.set_title("Regularized models on the test sample")
ax.legend(frameon=False, ncol=3)
ax.grid(alpha=0.25)
plt.tight_layout()

## Wrap-up: the results table

Everything estimated is saved in `output/workshop_results.csv`. The file fills up over the
three blocks, in Python and in R, so in Block 3 you can compare the LSTM with these models
without re-estimating them.

Things to discuss: how far do the penalized models get below OLS? And how far below the two
naive benchmarks? On 12 noisy test quarters, a gain of a tenth of a point is within what a
different test window could reverse.

In [ ]:
results_table();